# E5 — Demo: Pipeline completo E2→E3→E4

Notebook de demostración end-to-end del pipeline de reparación 3D:

```
[Vóxeles E2]  ──►  convertir_voxels_a_nube  ──►  [Nube rota (2048,3)]
                                                         │
                                                         ▼
                                              PoinTr  (E3 shape completion)
                                                         │
                                                         ▼
                                             [Nube completa (2048+N,3)]
                                                         │
                                                         ▼
                                        Poisson + manifold3d  (E4)
                                                         │
                                                         ▼
                                                 [STL imprimible]
```

**Modo de uso:**
- **Con vóxeles reales de E2**: sube el fichero `.npy` de salida de Pix2Vox++ → la Celda 3 lo convierte.
- **Sin E2 todavía** (demo): la Celda 3 usa una nube de puntos sintética del dataset de test.

Al final, la Celda 9 lanza una **aplicación Gradio** accesible desde el navegador.

---
## Sección 1 — Instalación y clonado

> ⚠️ **IMPORTANTE — flujo correcto:**
> 1. Ejecuta esta sección **una sola vez**
> 2. Cuando termine, **reinicia el runtime**: Entorno de ejecución → Reiniciar sesión
> 3. Después ejecuta directamente **Sección 2 → 2b → 7**

**E1 (eliminación de fondo)** usa OpenCV (ya incluido) — no necesita rembg ni dependencias extra.

In [ ]:
import subprocess, os
from getpass import getpass

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'gradio',
            'numpy', 'easydict', 'timm', 'opencv-python-headless']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

# Clonar repos
if not os.path.exists('/content/PoinTr'):
    subprocess.run(['git','clone','https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr','--depth=1','-q'], capture_output=True)
    print('PoinTr clonado.')

REPO = '/content/TFM'
if not os.path.exists(REPO):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO,'-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

os.chdir(REPO)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)

print()
print('=' * 60)
print('  PASO OBLIGATORIO: REINICIA EL RUNTIME AHORA')
print('  Menú: Entorno de ejecución → Reiniciar sesión')
print()
print('  Después ejecuta SOLO:')
print('    → Sección 2  (cargar PoinTr)')
print('    → Sección 2b (cargar Pix2Vox++)')
print('    → Sección 7  (lanzar app)')
print('  NO vuelvas a ejecutar esta sección.')
print('=' * 60)

---
## Sección 2 — Cargar modelo PoinTr (E3)

In [ ]:
import sys, types, glob as _glob, importlib
import torch, torch.nn as nn, numpy as np
from pathlib import Path
from easydict import EasyDict
from google.colab import drive

# ── Fix: PIL._typing._Ink faltante (Pillow < 10.4 vs Python 3.13) ─────────────
# rembg puede degradar Pillow a una versión sin _Ink; este parche lo añade en memoria
import PIL._typing as _pil_t
if not hasattr(_pil_t, '_Ink'):
    _pil_t._Ink = int | tuple[int, ...]
    print('[fix] PIL._typing._Ink parcheado (Pillow < 10.4 detectado)')

if not (Path('/content/drive').exists() and list(Path('/content/drive').iterdir())):
    drive.mount('/content/drive')
    print('Drive montado.')

DRIVE      = '/content/drive/MyDrive'
VERSION_E3 = 'v6_obj_sn'
BASE_E3    = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
DRIVE_E5   = f'{DRIVE}/E5'

for p in ['/content/TFM', '/content/PoinTr']:
    if p in sys.path: sys.path.remove(p)
for p in [DRIVE_E5, '/content/TFM', '/content/PoinTr']:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
os.chdir('/content/TFM')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')
print(f'Checkpoint : {BASE_E3}/modelos/{VERSION_E3}/best.pt')
print(f'Script conv: {DRIVE_E5}/convertir_voxels_a_nube.py')

_pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
_del = [k for k,v in sys.modules.items()
        if any(k==p or k.startswith(p+'.') for p in _pfx)
        or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
for k in _del: del sys.modules[k]
importlib.invalidate_caches()

for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
    try:
        src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
        if new!=src: open(fp,'w',encoding='utf-8').write(new)
    except: pass

def _force(n,a):
    m=types.ModuleType(n)
    for k,v in a.items(): setattr(m,k,v)
    sys.modules[n]=m

def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
class _CL1(nn.Module):
    def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
_ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL1,'ChamferDistanceL1_PM':_CL1,'chamfer_3DDist':_cr}
for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']:
    _force(n,_ch)

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz,np_):
        B,N,_=xyz.shape; dev=xyz.device
        idx=torch.zeros(B,np_,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
        far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
        for i in range(np_):
            idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
            dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
        return idx
    def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
    def _bq(r,ns,xyz,nxyz):
        d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
        return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
    def _grp(f,idx):
        B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
        return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
    def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
    def _3i(f,idx,w):
        B,C,M=f.shape; N=idx.shape[1]
        return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
    _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
    _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
    sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu

class _KNN(nn.Module):
    def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
    def forward(self,ref,query):
        if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
        r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
        d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
        return dk.transpose(1,2),ik.transpose(1,2)
_force('knn_cuda',{'KNN':_KNN})

def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
class _Emd(nn.Module):
    def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                   ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                   ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                   ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
    for pfx in ['','extensions.']:
        if pfx+base not in sys.modules: _force(pfx+base,attrs)

ckpt_local = f'E3/checkpoints_pointr_{VERSION_E3}/best.pt'
ckpt_drive  = f'{BASE_E3}/modelos/{VERSION_E3}/best.pt'
ckpt_path   = ckpt_local if Path(ckpt_local).exists() else ckpt_drive
print(f'Cargando checkpoint desde: {ckpt_path}')
ck = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'PoinTr {VERSION_E3} — best epoch {ck["epoch"]}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ck['model_cfg']))
except:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ck['model_cfg']))
model.load_state_dict(ck['model_state_dict'])
model = model.to(device).eval()
print(f'Modelo listo. Params: {sum(p.numel() for p in model.parameters()):,}')

try:
    from convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels
    print('[OK] Script de conversión cargado desde Drive/E5')
except ImportError:
    from Scripts.convertir_voxels_a_nube import voxels_a_nube, diagnosticar_voxels
    print('[OK] Script de conversión cargado desde el repo')

---
## Sección 2b — Cargar modelo Pix2Vox++ (E2)

In [ ]:
import cv2 as _cv2
import sys as _sys

_e2_path = '/content/TFM/E2'
if _e2_path not in _sys.path:
    _sys.path.insert(0, _e2_path)

from modelo_pix2vox import Pix2VoxPlusPlusA, cargar_checkpoint_finetuning

CHECKPOINT_E2 = 'exp13_7categorias_5v_finetuning_mejor.pth'
RUTA_CKPT_E2_DRIVE = f'{DRIVE}/Datos_E2_E3/E2/Pix2Vox++/checkpoints/{CHECKPOINT_E2}'
RUTA_CKPT_E2_LOCAL = f'/content/TFM/E2/checkpoints/{CHECKPOINT_E2}'

ruta_ckpt_e2 = RUTA_CKPT_E2_LOCAL if Path(RUTA_CKPT_E2_LOCAL).exists() else RUTA_CKPT_E2_DRIVE
print(f'Cargando Pix2Vox++ desde: {ruta_ckpt_e2}')

modelo_e2 = Pix2VoxPlusPlusA(usar_refiner=True, usar_merger=True, usar_pesos_imagenet=False)
meta_e2 = cargar_checkpoint_finetuning(modelo_e2, ruta_ckpt_e2, dispositivo_carga='cpu')
modelo_e2 = modelo_e2.to(device).eval()
UMBRAL_E2 = meta_e2.get('mejor_umbral') or 0.3
print(f'Pix2Vox++ listo | época={meta_e2.get("epoca")} | umbral={UMBRAL_E2}')

# Fondo blanco puro (1.0) = igual que los renders de ShapeNet usados en entrenamiento
_FONDO_E2 = 1.0

def _cargar_img_e2(ruta):
    img = _cv2.imread(str(ruta), _cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'No se pudo cargar: {ruta}')
    if img.ndim == 2:
        img = _cv2.cvtColor(img, _cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        alpha = img[:, :, 3:4].astype(np.float32) / 255.0
        bgr   = img[:, :, :3].astype(np.float32) / 255.0
        bgr   = bgr * alpha + _FONDO_E2 * (1 - alpha)
    else:
        bgr = img[:, :, :3].astype(np.float32) / 255.0
    bgr = _cv2.resize(bgr, (224, 224), interpolation=_cv2.INTER_LINEAR)
    bgr = (bgr - 0.5) / 0.5
    return torch.tensor(bgr.transpose(2, 0, 1), dtype=torch.float32)

@torch.no_grad()
def inferir_voxel_e2(rutas_imagenes):
    tensors = [_cargar_img_e2(r) for r in rutas_imagenes]
    imgs = torch.stack(tensors).unsqueeze(0).to(device)  # (1,5,3,224,224)
    return modelo_e2(imgs)['volumen_final'].squeeze().cpu().numpy()

print('Funciones E2 listas (inferir_voxel_e2).')

---
## Sección 3 — Cargar nube rota (desde E2 o demo sintética)

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPCION A — Vóxeles de E2 (Pix2Vox++): subir fichero .npy
# ══════════════════════════════════════════════════════════════
USAR_VOXELES_E2 = False  # ← cambia a True si tienes salida de Pix2Vox++

if USAR_VOXELES_E2:
    from google.colab import files
    print('Sube el fichero .npy de vóxeles de E2 (shape 32×32×32):')
    subidos = files.upload()
    ruta_voxels = list(subidos.keys())[0]

    # voxels_a_nube y diagnosticar_voxels ya fueron importados en la Sección 2
    voxels = np.load(ruta_voxels)
    diag = diagnosticar_voxels(voxels)
    print(f'Grid: {diag["forma_grid"]}  ocupado: {diag["porcentaje_ocupado"]}%')
    for w in diag['advertencias']:
        if w: print(f'  ⚠️  {w}')

    nube_rota = voxels_a_nube(voxels, n_puntos=2048)
    NOMBRE_OBJETO = Path(ruta_voxels).stem
    print(f'Nube generada desde vóxeles: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')

else:
    # ══════════════════════════════════════════════════════════
    # OPCION B — Demo: usar una nube del test set de E3
    # ══════════════════════════════════════════════════════════
    import E3.dataset as _ds; _ds.CENTRAR_EN_ROTO = False
    from E3.dataset import construir_pares
    import random

    BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'
    carpetas = [f'{BASE_GEN}/shapenet_roturas', f'{BASE_GEN}/roturas_Objaverse_v2']
    carpetas_ok = [c for c in carpetas if Path(c).exists()]

    if not carpetas_ok:
        carpetas_ok = [c for c in ['Datos/shapenet/roturas', 'Datos/objaverse/roturas_v2']
                       if Path(c).exists()]

    todos = construir_pares(carpetas_ok)
    rng = random.Random(42); rng.shuffle(todos)
    n = len(todos); _te = todos[int(0.9*n):]

    INDICE_DEMO = 0   # ← cambia para ver otros objetos del test set
    ruta_roto, ruta_comp = _te[INDICE_DEMO]
    nube_rota = np.load(ruta_roto).astype(np.float32)
    nube_gt   = np.load(ruta_comp).astype(np.float32)
    NOMBRE_OBJETO = Path(ruta_roto).stem.replace('_roto','')
    print(f'Objeto de demo: {NOMBRE_OBJETO}')
    print(f'Nube rota: {nube_rota.shape}  radio_max={np.linalg.norm(nube_rota,axis=1).max():.3f}')
    print(f'(GT disponible para comparación — no se usa como entrada al modelo)')

---
## Sección 4 — E3: PoinTr shape completion

In [ ]:
def inferir_pointr(nube_np: np.ndarray) -> np.ndarray:
    """Nube rota (2048,3) → nube completa predicha (N,3)."""
    with torch.no_grad():
        inp = torch.tensor(nube_np, dtype=torch.float32).unsqueeze(0).to(device)
        out = model(inp)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        return fine.squeeze(0).cpu().numpy()

def filtrar_outliers(pts, k=20, std_ratio=1.5):
    """Filtra outliers por distancia media a k vecinos. Usa torch, sin scipy."""
    pts_t = torch.tensor(pts, dtype=torch.float32)
    dists = torch.cdist(pts_t.unsqueeze(0), pts_t.unsqueeze(0)).squeeze(0)
    topk  = dists.topk(k + 1, dim=1, largest=False).values[:, 1:]
    mean_d = topk.mean(dim=1).cpu().numpy()
    umbral = float(mean_d.mean() + std_ratio * mean_d.std())
    mask = mean_d < umbral
    return pts[mask], int((~mask).sum())

def chamfer_l1(p, g):
    pt = torch.tensor(p).unsqueeze(0); gt = torch.tensor(g).unsqueeze(0)
    d = torch.cdist(pt, gt, p=2)
    return ((d.min(2).values.mean() + d.min(1).values.mean()) / 2).item()

# Inferencia E3
print('Ejecutando PoinTr...')
pred_raw = inferir_pointr(nube_rota)
pred, n_outliers = filtrar_outliers(pred_raw)
print(f'PoinTr: {pred_raw.shape[0]} pts → {pred.shape[0]} pts (eliminados {n_outliers} outliers)')

# Métricas (solo si tenemos GT)
if 'nube_gt' in locals():
    cd_val = chamfer_l1(pred, nube_gt)
    print(f'CD-L1 vs GT: {cd_val:.4f}')

# Visualización
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pred[:,0], y=pred[:,1], z=pred[:,2], mode='markers',
    name='PoinTr (completo)', marker=dict(size=2, color='#66BB6A', opacity=0.85)))
fig.add_trace(go.Scatter3d(x=nube_rota[:,0], y=nube_rota[:,1], z=nube_rota[:,2], mode='markers',
    name='Roto (entrada)', marker=dict(size=3, color='#EF5350', opacity=0.95)))
fig.update_layout(
    scene=dict(bgcolor='#111',
               xaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               yaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               zaxis=dict(showticklabels=False, backgroundcolor='#111', gridcolor='#333'),
               aspectmode='cube'),
    title=dict(text=f'<b>E3: Shape completion</b> — {NOMBRE_OBJETO}<br>'
                    '<sup>Rojo=roto | Verde=reconstruido por PoinTr</sup>',
               font=dict(color='white'), x=0.5),
    paper_bgcolor='#111', legend=dict(font=dict(color='white')),
    height=560, width=680, margin=dict(l=0,r=0,t=60,b=0))
fig.show()
print('E3 completado.')

---
## Sección 5 — E4: Generación de STL

In [ ]:
import pymeshlab, trimesh

# ─────────────────────────────────────────────────────────────────────────────
# NOTA: estas funciones son para DEBUG paso a paso (Secciones 3-6).
# La app Gradio (Sección 7) tiene su propia versión mejorada e integrada.
# ─────────────────────────────────────────────────────────────────────────────

TAMANO_MM = 100.0

def suavizar_nube(pts, k=15, iters=3):
    """Suavizado Laplaciano con torch (sin scipy.cKDTree)."""
    pts_t = torch.tensor(pts, dtype=torch.float32)
    dists = torch.cdist(pts_t.unsqueeze(0), pts_t.unsqueeze(0)).squeeze(0)
    idx = dists.topk(k + 1, dim=1, largest=False).indices[:, 1:]
    result = pts_t
    for _ in range(iters):
        result = result[idx].mean(dim=1)
    return result.cpu().numpy().astype(np.float32)

def poisson_stl(pts, depth=8):
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pts.astype(np.float64)))
    ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
    ms.generate_surface_reconstruction_screened_poisson(depth=depth, scale=1.1)
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=200)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)
    m = ms.current_mesh()
    return trimesh.Trimesh(vertices=m.vertex_matrix(), faces=m.face_matrix(), process=False)

def reparar(mesh):
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda c: len(c.faces))
    try:
        import manifold3d
        m = manifold3d.Manifold(manifold3d.Mesh(
            vert_properties=np.array(mesh.vertices, dtype=np.float32),
            tri_verts=np.array(mesh.faces, dtype=np.uint32)))
        out = m.to_mesh()
        res = trimesh.Trimesh(vertices=np.array(out.vert_properties),
                              faces=np.array(out.tri_verts), process=False)
        if len(res.vertices) > 0:
            mesh = res
    except Exception as e:
        print(f'  manif3d falló: {e}')
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.process(validate=False)
    if not mesh.is_watertight and abs(int(mesh.euler_number)) > 4:
        try:
            hull = mesh.convex_hull
            if hull.is_watertight:
                mesh = hull
                print('  → convex hull aplicado como fallback (watertight logrado)')
        except Exception as e:
            print(f'  convex hull falló: {e}')
    return mesh

# Pipeline E4
print('Ejecutando E4...')
pred_suav = suavizar_nube(pred, k=15, iters=3)
print(f'  Suavizado Laplaciano: {pred.shape[0]} pts')

mesh_raw = poisson_stl(pred_suav, depth=8)
print(f'  Poisson depth=8: {len(mesh_raw.faces):,} caras')

mesh_raw.apply_translation(-mesh_raw.centroid)
lado = mesh_raw.bounding_box.extents.max()
if lado > 0: mesh_raw.apply_scale(TAMANO_MM / lado)

mesh_final = reparar(mesh_raw)
wt = bool(mesh_final.is_watertight)
eu = int(mesh_final.euler_number)
print(f'  Reparación: {len(mesh_final.faces):,} caras | watertight={wt} | euler={eu}')
print()
print('═'*50)
print(f'RESULTADO: {"WATERTIGHT" if wt else "No watertight"}')
print(f'  Caras   : {len(mesh_final.faces):,}')
print(f'  Euler   : {eu}  (2=sólido, 0=taza sin asa, -2=taza con asa)')
print(f'  Tamaño  : {TAMANO_MM:.0f} mm lado mayor')
print('═'*50)

# Guardar STL
Path('E5').mkdir(exist_ok=True)
ruta_stl = f'E5/{NOMBRE_OBJETO}_demo.stl'
mesh_final.export(ruta_stl)
print(f'STL guardado: {ruta_stl}')

# Visualización STL
verts = np.array(mesh_final.vertices)
faces = np.array(mesh_final.faces)
if len(faces) > 0:
    z = verts[:,2]
    intensidad = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig2 = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#0D47A1'],[0.5,'#29B6F6'],[1,'#E1F5FE']],
        showscale=False,
        lighting=dict(ambient=0.3,diffuse=0.85,roughness=0.3,specular=0.6),
        lightposition=dict(x=200,y=300,z=400)))
    fig2.update_layout(
        scene=dict(bgcolor='#111',
                   xaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   yaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   zaxis=dict(showticklabels=False,backgroundcolor='#111',gridcolor='#333'),
                   aspectmode='data'),
        title=dict(text=f'E4: STL final — {NOMBRE_OBJETO} | '
                        f'{"WATERTIGHT" if wt else "no watertight"} | euler={eu} | {len(faces):,} caras',
                   font=dict(color='white'), x=0.5),
        paper_bgcolor='#111', height=560, width=680,
        margin=dict(l=0,r=0,t=60,b=0))
    fig2.show()

---
## Sección 6 — Descargar STL

In [ ]:
from google.colab import files
if Path(ruta_stl).exists():
    print(f'Descargando {ruta_stl}...')
    files.download(ruta_stl)
else:
    print('STL no encontrado — ejecuta la Sección 5 primero.')

---
## Seccion 7 — App REBUILD3D

> **Flujo de uso:**
> 1. Ejecutar **Sección 1** (instalar librerías)
> 2. Ejecutar **Sección 2** y **Sección 2b** (cargar modelos PoinTr y Pix2Vox++)
> 3. Ejecutar **esta celda** → aparece enlace `*.gradio.live` → abrirlo en el navegador
>
> Las Secciones 3–6 son opcionales (debug paso a paso). No hace falta ejecutarlas para usar la app.

**Tab 1 — Pipeline completo:** sube 5 fotos → E1 (rembg, quita fondo) → E2 (Pix2Vox++) → E3 (PoinTr) → E4 (STL imprimible)

**Tab 2 — Desde .npy:** voxeles (32×32×32) o nube de puntos (N×3) → E3 → E4

Cada etapa muestra una **visualizacion 3D interactiva** en su propia pestana.

**Novedades v2:**
- E1 rembg integrado: el fondo se elimina automaticamente antes de pasar a E2
- E4 Poisson depth=8 (más detalle que depth=7)
- Reparacion mejorada: manifold3d con log de errores + convex hull como fallback
- Log con % de voxeles activos y aviso si son ruidosos (>30%)

In [ ]:
import gradio as gr
import tempfile
import cv2 as _cv2
import pymeshlab, trimesh
import plotly.graph_objects as go
# SIN scipy — todo con torch/numpy para evitar conflicto numpy._core.umath._slice

# ── Helpers de visualización 3D ───────────────────────────────────────────────
_BG = '#0F1117'
_AXIS_STYLE = dict(showticklabels=False, backgroundcolor=_BG,
                   gridcolor='#2D3250', showspikes=False, zeroline=False)
_SCENE_BASE = dict(bgcolor=_BG, xaxis=_AXIS_STYLE, yaxis=_AXIS_STYLE, zaxis=_AXIS_STYLE,
                   aspectmode='cube', camera=dict(eye=dict(x=1.4, y=1.4, z=0.8)))
_LAYOUT_BASE = dict(paper_bgcolor=_BG, plot_bgcolor=_BG,
                    legend=dict(font=dict(color='#CFD8DC', size=12), bgcolor='rgba(0,0,0,0.4)',
                                bordercolor='#37474F', borderwidth=1),
                    margin=dict(l=0, r=0, t=48, b=0), height=430, font=dict(color='#CFD8DC'))

def _placeholder_fig(msg='Resultado aparecerá aquí tras ejecutar el pipeline'):
    return go.Figure(layout=go.Layout(paper_bgcolor=_BG, plot_bgcolor=_BG, height=430,
        annotations=[dict(text=msg, showarrow=False, font=dict(color='#455A64', size=14),
                          xref='paper', yref='paper', x=0.5, y=0.5)]))

def _fig_voxeles(voxel_prob, umbral=0.3, titulo=''):
    voxel_bin = voxel_prob >= umbral
    zz, yy, xx = np.where(voxel_bin)
    if len(xx) == 0:
        return _placeholder_fig('Sin voxeles activos')
    coords = np.stack([xx, yy, zz], axis=1).astype(float)
    coords = (coords / (voxel_prob.shape[0] - 1)) * 2 - 1
    fig = go.Figure(go.Scatter3d(x=coords[:,0], y=coords[:,1], z=coords[:,2], mode='markers',
        name=f'Voxeles activos ({voxel_bin.sum():,} / {voxel_prob.size:,})',
        marker=dict(size=3.5, color=voxel_prob[voxel_bin],
                    colorscale=[[0,'#1565C0'],[0.5,'#00BCD4'],[1,'#80DEEA']],
                    opacity=0.8, showscale=True,
                    colorbar=dict(title=dict(text='P(ocupado)', font=dict(color='#CFD8DC')),
                                  tickfont=dict(color='#CFD8DC'), thickness=12, len=0.6))))
    fig.update_layout(scene=_SCENE_BASE,
                      title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
                      **_LAYOUT_BASE)
    return fig

def _fig_nube(roto, completo=None, titulo=''):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=roto[:,0], y=roto[:,1], z=roto[:,2], mode='markers',
        name='Entrada (roto)', marker=dict(size=2.5, color='#EF5350', opacity=0.9)))
    if completo is not None:
        fig.add_trace(go.Scatter3d(x=completo[:,0], y=completo[:,1], z=completo[:,2],
            mode='markers', name='Reconstruido', marker=dict(size=2, color='#00BCD4', opacity=0.75)))
    fig.update_layout(scene=_SCENE_BASE,
                      title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
                      **_LAYOUT_BASE)
    return fig

def _fig_mesh(mesh, titulo=''):
    verts = np.array(mesh.vertices); faces = np.array(mesh.faces)
    if len(faces) == 0:
        return _placeholder_fig('Sin malla generada')
    z = verts[:,2]; intens = (z - z.min()) / (np.ptp(z) + 1e-8)
    fig = go.Figure(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2], intensity=intens,
        colorscale=[[0,'#1565C0'],[0.45,'#00BCD4'],[1,'#E0F7FA']], showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.9, roughness=0.3, specular=0.7),
        lightposition=dict(x=200, y=300, z=400)))
    fig.update_layout(scene={**_SCENE_BASE, 'aspectmode': 'data'},
                      title=dict(text=titulo, font=dict(color='#CFD8DC', size=13), x=0.5),
                      **_LAYOUT_BASE)
    return fig

# ── E1: Eliminación de fondo con OpenCV ───────────────────────────────────────
def _quitar_fondo(ruta_img):
    try:
        img = _cv2.imread(str(ruta_img))
        if img is None: return ruta_img, False
        lab = _cv2.cvtColor(img, _cv2.COLOR_BGR2LAB)
        L = lab[:, :, 0]
        umbral = max(int(np.percentile(L, 75)), 155)
        _, mask = _cv2.threshold(L, umbral, 255, _cv2.THRESH_BINARY_INV)
        k = _cv2.getStructuringElement(_cv2.MORPH_ELLIPSE, (7, 7))
        mask = _cv2.morphologyEx(mask, _cv2.MORPH_CLOSE, k, iterations=4)
        mask = _cv2.morphologyEx(mask, _cv2.MORPH_OPEN,  k, iterations=2)
        n_labels, labels, stats, _ = _cv2.connectedComponentsWithStats(mask)
        if n_labels <= 1: return ruta_img, False
        largest = 1 + int(np.argmax(stats[1:, _cv2.CC_STAT_AREA]))
        obj_mask = (labels == largest).astype(np.uint8) * 255
        result = np.full_like(img, 255)
        result[obj_mask > 0] = img[obj_mask > 0]
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False, prefix='e1_cv_')
        _cv2.imwrite(tmp.name, result)
        return tmp.name, True
    except Exception:
        return ruta_img, False

# ── E2: Limpieza adaptativa de voxeles ruidosos (sin scipy) ──────────────────
def _limpiar_voxels_e2(voxels_prob, umbral_ini=0.3):
    umbral_final = umbral_ini
    for u in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        if u < umbral_ini - 0.01:
            continue
        mask = voxels_prob >= u
        pct = 100.0 * float(mask.sum()) / mask.size
        umbral_final = u
        if pct <= 20.0:
            break
    binario = (voxels_prob >= umbral_final)
    b = binario.astype(np.int8)
    nbrs = (np.roll(b, 1, 0) + np.roll(b, -1, 0) +
            np.roll(b, 1, 1) + np.roll(b, -1, 1) +
            np.roll(b, 1, 2) + np.roll(b, -1, 2))
    eroded = binario & (nbrs >= 3)
    if int(eroded.sum()) < 200:
        eroded = binario
    pct_final = round(100.0 * float(eroded.sum()) / eroded.size, 1)
    return eroded.astype(np.float32), umbral_final, pct_final

# ── E3: filtrar outliers con torch (sin scipy) ────────────────────────────────
def _filtrar_outliers(pts, k=20, std_ratio=1.5):
    pts_t = torch.tensor(pts, dtype=torch.float32)
    dists = torch.cdist(pts_t.unsqueeze(0), pts_t.unsqueeze(0)).squeeze(0)
    topk  = dists.topk(k + 1, dim=1, largest=False).values[:, 1:]
    mean_d = topk.mean(dim=1).cpu().numpy()
    umbral = float(mean_d.mean() + std_ratio * mean_d.std())
    mask = mean_d < umbral
    return pts[mask], int((~mask).sum())

# ── E4: Reparación con fallback a convex hull ─────────────────────────────────
def _reparar(mesh):
    avisos = []
    comps = mesh.split(only_watertight=False)
    if len(comps) > 1:
        mesh = max(comps, key=lambda c: len(c.faces))
    try:
        import manifold3d
        m = manifold3d.Manifold(manifold3d.Mesh(
            vert_properties=np.array(mesh.vertices, dtype=np.float32),
            tri_verts=np.array(mesh.faces, dtype=np.uint32)))
        out = m.to_mesh()
        res = trimesh.Trimesh(vertices=np.array(out.vert_properties),
                              faces=np.array(out.tri_verts), process=False)
        if len(res.vertices) > 0: mesh = res
    except Exception as e:
        avisos.append(f'  manif3d: {str(e)[:60]}')
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.process(validate=False)
    if not mesh.is_watertight and abs(int(mesh.euler_number)) > 4:
        try:
            hull = mesh.convex_hull
            if hull.is_watertight:
                mesh = hull
                avisos.append('  convex hull aplicado (watertight logrado)')
        except Exception as e:
            avisos.append(f'  convex hull: {str(e)[:50]}')
    return mesh, avisos

def _interpretar_euler(eu):
    if eu == 2: return 'solido simple'
    if eu == 0: return 'taza sin asa'
    if eu == -2: return 'taza con asa'
    return 'topologia compleja'

# ── Pipeline interno (torch puro, sin scipy) ──────────────────────────────────
def _suavizar_torch(pts, k=15, iters=3):
    pts_t = torch.tensor(pts, dtype=torch.float32)
    dists = torch.cdist(pts_t.unsqueeze(0), pts_t.unsqueeze(0)).squeeze(0)
    idx = dists.topk(k + 1, dim=1, largest=False).indices[:, 1:]
    result = pts_t
    for _ in range(iters):
        result = result[idx].mean(dim=1)
    return result.cpu().numpy().astype(np.float32)

def _run_e3_e4(nube_rota):
    log = []
    pred_raw = inferir_pointr(nube_rota)
    pred, n_out = _filtrar_outliers(pred_raw)
    log.append(f'E3 PoinTr: {pred.shape[0]} pts (eliminados {n_out} outliers)')
    fig_e3 = _fig_nube(nube_rota, pred, 'E3 — Shape completion  |  rojo=roto · cyan=reconstruido')

    pred_suav = _suavizar_torch(pred, k=15, iters=3)
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pred_suav.astype(np.float64)))
    ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
    ms.generate_surface_reconstruction_screened_poisson(depth=8, scale=1.1)
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=200)
    ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)
    m = ms.current_mesh()
    mesh_raw = trimesh.Trimesh(vertices=m.vertex_matrix(), faces=m.face_matrix(), process=False)
    log.append(f'  Poisson depth=8: {len(mesh_raw.faces):,} caras')

    mesh_raw.apply_translation(-mesh_raw.centroid)
    lado = mesh_raw.bounding_box.extents.max()
    if lado > 0: mesh_raw.apply_scale(100.0 / lado)

    mesh_final, avisos = _reparar(mesh_raw)
    log.extend(avisos)
    wt  = bool(mesh_final.is_watertight)
    eu  = int(mesh_final.euler_number)
    estado = 'WATERTIGHT' if wt else 'no watertight'
    log.append(f'E4 STL: {len(mesh_final.faces):,} caras | {estado} | euler={eu} ({_interpretar_euler(eu)}) | 100mm')
    fig_e4 = _fig_mesh(mesh_final, f'E4 — Malla STL  |  {len(mesh_final.faces):,} caras  [{estado}]')
    tmp = tempfile.NamedTemporaryFile(suffix='.stl', delete=False, prefix='rebuild3d_')
    mesh_final.export(tmp.name)
    return tmp.name, '\n'.join(log), fig_e3, fig_e4

# ── Funciones Gradio ──────────────────────────────────────────────────────────
_PH = _placeholder_fig()

def pipeline_imagenes(img1, img2, img3, img4, img5):
    imgs = [img1, img2, img3, img4, img5]
    if any(x is None for x in imgs):
        return None, 'Sube exactamente 5 imagenes (una por vista)', _PH, _PH, _PH, _PH
    try:
        rutas_orig = [x if isinstance(x, str) else x.name for x in imgs]
        rutas_limpias, n_ok = [], 0
        for r in rutas_orig:
            r_clean, ok = _quitar_fondo(r)
            rutas_limpias.append(r_clean)
            if ok: n_ok += 1
        if n_ok == 5:   log = ['E1 OpenCV: fondo eliminado en las 5 imagenes']
        elif n_ok == 0: log = ['E1 OpenCV: fallo — usando imagenes originales']
        else:           log = [f'E1 OpenCV: fondo eliminado en {n_ok}/5 imagenes']

        voxels = inferir_voxel_e2(rutas_limpias)
        umbral_ini = UMBRAL_E2 if 'UMBRAL_E2' in dir() else 0.3
        pct_ini = round(100 * float((voxels >= umbral_ini).sum()) / voxels.size, 1)
        voxels_clean, umbral_usado, pct_final = _limpiar_voxels_e2(voxels, umbral_ini)
        aviso = '  RUIDOSO' if pct_final > 25 else ''
        log.append(f'E2 Pix2Vox++: grid {voxels.shape}  umbral {umbral_ini:.2f}({pct_ini}%) → {umbral_usado:.2f}({pct_final}%){aviso}')
        fig_vox = _fig_voxeles(voxels_clean, 0.5, f'E2 — Voxeles 32x32x32  ({pct_final}% activos tras limpieza)')

        nube_rota = voxels_a_nube(voxels_clean, n_puntos=2048, umbral=0.5)
        log.append(f'  Conversion voxel → nube: {nube_rota.shape}')
        fig_nube  = _fig_nube(nube_rota, titulo='E2 — Nube de puntos (2048 pts)')

        stl_path, log_rest, fig_e3, fig_e4 = _run_e3_e4(nube_rota)
        return stl_path, '\n'.join(log) + '\n' + log_rest, fig_vox, fig_nube, fig_e3, fig_e4
    except Exception as e:
        import traceback
        return None, f'Error:\n{e}\n\n{traceback.format_exc()}', _PH, _PH, _PH, _PH

def pipeline_npy(archivo_npy):
    if archivo_npy is None:
        return None, 'Sube un fichero .npy primero', _PH, _PH
    try:
        arr = np.load(archivo_npy.name).astype(np.float32)
        while arr.ndim > 3 and arr.shape[0] == 1: arr = arr[0]
        if arr.ndim == 3:
            diag = diagnosticar_voxels(arr)
            nube_rota = voxels_a_nube(arr, n_puntos=2048)
            log = [f'Voxeles {diag["forma_grid"]}  ({diag["porcentaje_ocupado"]}% ocupado) → nube {nube_rota.shape}']
        elif arr.ndim == 2 and arr.shape[1] == 3:
            # NO re-normalizar: los .npy del dataset ya están centrados en el objeto completo.
            # Re-centrar en la nube rota desplazaría el centroide y confundiría a PoinTr.
            if len(arr) == 2048:
                nube_rota = arr
                log = [f'Nube de puntos: {arr.shape}  (formato E3, sin re-normalizar)']
            else:
                # Si tiene más o menos puntos, subsamplear/completar y normalizar
                nube_rota = arr[np.random.choice(len(arr), 2048, replace=len(arr)<2048)]
                nube_rota = nube_rota - nube_rota.mean(0)
                r = np.linalg.norm(nube_rota, axis=1).max()
                if r > 1e-8: nube_rota = nube_rota / r
                nube_rota = nube_rota.astype(np.float32)
                log = [f'Nube de puntos: {arr.shape} → submuestreada/normalizada a (2048,3)']
        else:
            return None, f'Formato no reconocido: shape={arr.shape}', _PH, _PH
        stl_path, log_rest, fig_e3, fig_e4 = _run_e3_e4(nube_rota)
        return stl_path, '\n'.join(log) + '\n' + log_rest, fig_e3, fig_e4
    except Exception as e:
        import traceback
        return None, f'Error:\n{e}\n\n{traceback.format_exc()}', _PH, _PH

# ── CSS y cabecera ────────────────────────────────────────────────────────────
CSS = (
    '.rebuild3d-header{background:linear-gradient(135deg,#0A1628 0%,#0D1F3C 60%,#091524 100%);'
    'border:1px solid #1E3A5F;border-bottom:2px solid #00BCD4;'
    'padding:28px 36px 20px;border-radius:14px;margin-bottom:4px;}'
    '.rebuild3d-logo{font-size:2.6em;font-weight:900;letter-spacing:.15em;'
    'background:linear-gradient(90deg,#00BCD4 0%,#4FC3F7 50%,#80DEEA 100%);'
    '-webkit-background-clip:text;-webkit-text-fill-color:transparent;'
    'background-clip:text;margin:0 0 4px 0;line-height:1;}'
    '.rebuild3d-tagline{color:#546E7A;font-size:.88em;letter-spacing:.06em;margin:0;}'
    '.rebuild3d-pipeline{display:flex;gap:8px;align-items:center;margin-top:10px;flex-wrap:wrap;}'
    '.rb-stage{background:rgba(0,188,212,.1);border:1px solid rgba(0,188,212,.3);'
    'color:#80DEEA;font-size:.78em;font-weight:600;padding:3px 12px;border-radius:20px;letter-spacing:.05em;}'
    '.rb-arrow{color:#37474F;font-size:1em;}'
    '.log-mono textarea{font-family:"JetBrains Mono","Fira Code",monospace!important;'
    'font-size:.82em!important;background:#0A0E18!important;color:#90A4AE!important;'
    'border:1px solid #1E2D3D!important;}'
    'footer{display:none!important;}'
)
HEADER_HTML = (
    '<div class="rebuild3d-header">'
    '<p class="rebuild3d-logo">REBUILD3D</p>'
    '<p class="rebuild3d-tagline">Reconstruccion y reparacion de objetos 3D rotos &middot; TFM UCM 2026</p>'
    '<div class="rebuild3d-pipeline">'
    '<span class="rb-stage">E1 seg.fondo</span><span class="rb-arrow">&rarr;</span>'
    '<span class="rb-stage">E2 Pix2Vox++</span><span class="rb-arrow">&rarr;</span>'
    '<span class="rb-stage">E3 PoinTr</span><span class="rb-arrow">&rarr;</span>'
    '<span class="rb-stage">E4 STL imprimible</span>'
    '</div></div>'
)

with gr.Blocks(title='REBUILD3D',
               theme=gr.themes.Soft(primary_hue=gr.themes.colors.cyan,
                                    neutral_hue=gr.themes.colors.slate,
                                    font=[gr.themes.GoogleFont('Inter'), 'sans-serif']),
               css=CSS) as app:

    gr.HTML(HEADER_HTML)

    with gr.Tabs():
        with gr.Tab('Pipeline completo (imagenes)'):
            gr.Markdown('**Sube 5 fotos** del objeto roto. El fondo se elimina con OpenCV (E1). '
                        'Orden: frontal, lateral x2, trasera, superior.')
            with gr.Row():
                imgs_in = [gr.Image(type='filepath', label=f'Vista {i+1}', height=150) for i in range(5)]
            btn1 = gr.Button('Reconstruir desde imagenes', variant='primary', size='lg')
            with gr.Row():
                with gr.Column(scale=1, min_width=280):
                    stl_out1 = gr.File(label='STL generado', file_types=['.stl'])
                    log_out1 = gr.Textbox(label='Log', lines=14, interactive=False, elem_classes=['log-mono'])
                with gr.Column(scale=2):
                    with gr.Tabs():
                        with gr.Tab('E2 — Voxeles'):   plot_vox = gr.Plot(value=_PH)
                        with gr.Tab('E2 — Nube'):      plot_e2  = gr.Plot(value=_PH)
                        with gr.Tab('E3 — Completion'):plot_e3a = gr.Plot(value=_PH)
                        with gr.Tab('E4 — STL'):       plot_e4a = gr.Plot(value=_PH)
            btn1.click(fn=pipeline_imagenes, inputs=imgs_in,
                       outputs=[stl_out1, log_out1, plot_vox, plot_e2, plot_e3a, plot_e4a])

        with gr.Tab('Desde .npy (sin E2)'):
            gr.Markdown('Sube un `.npy` del dataset E3: nube rota (2048×3) o voxeles (32×32×32).')
            with gr.Row():
                with gr.Column(scale=1, min_width=280):
                    npy_in   = gr.File(label='Fichero .npy', file_types=['.npy'])
                    btn2     = gr.Button('Reconstruir desde .npy', variant='primary', size='lg')
                    stl_out2 = gr.File(label='STL generado', file_types=['.stl'])
                    log_out2 = gr.Textbox(label='Log', lines=14, interactive=False, elem_classes=['log-mono'])
                with gr.Column(scale=2):
                    with gr.Tabs():
                        with gr.Tab('E3 — Completion'): plot_e3b = gr.Plot(value=_PH)
                        with gr.Tab('E4 — STL'):        plot_e4b = gr.Plot(value=_PH)
            btn2.click(fn=pipeline_npy, inputs=[npy_in],
                       outputs=[stl_out2, log_out2, plot_e3b, plot_e4b])

app.launch(share=True, debug=False)
print('REBUILD3D lanzado -- abre el enlace de arriba.')